# PyRosetta 核心概念

对应笔记仓库 `037_Rosetta_Notes/Rosetta_Concepts.md`

## 01  Pose

Pose 是 Rosetta 表示「一个分子体系当前状态」的中心对象，里面装着：构象（xyz + 二面角）、序列与化学、链拓扑、上次打分的能量、以及 PDBInfo（原始 PDB 的链号与残基编号）。

两个要点：

1. **Pose 是可变的** —— 所有 Mover 都是原地修改传进去的 Pose，不返回新对象。
2. **两套残基编号** —— Rosetta 内部从 1 连续编到 N、无视链边界；PDB 文件按链分别计数。两者靠 PDBInfo 换算。

In [1]:
import pyrosetta

pyrosetta.init('-mute all')    # 每个进程只需 init 一次，加载数据库要十几秒

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python311.ubuntu 2026.29+releasequarterly.80a0635615099e1b918474a63acba7b1de6fd107 2026-07-14T16:24:11] retrieved from: http://www.pyrosetta.org


In [2]:
pose = pyrosetta.pose_from_sequence('AAAGGGKKK')    # 不用文件，直接从序列造一个 Pose

print(pose.total_residue())          # 残基总数
print(pose.sequence())               # 序列
print(pose.residue(3).name3())       # 第 3 个残基的三字母名（注意从 1 开始数）
print(pose.phi(3), pose.psi(3))      # 第 3 个残基的主链二面角

9
AAAGGGKKK
ALA
180.0 180.0


In [3]:
print(pose)    # Pose 的摘要：序列、折叠树、链信息

PDB file name: AAAGGGKK
Total residues: 9
Sequence: AAAGGGKKK
Fold tree:
FOLD_TREE  EDGE 1 9 -1 


### 01-2  加载真实复合物

`8hpu_M_N_A.pdb` 是一个抗体-抗原复合物：H 链（VH 121 aa）、L 链（VL 109 aa）、A 链（抗原 194 aa），共 424 残基。
只有 ATOM 记录，无水、无配体、无 altloc。

In [4]:
pdb = '/data/lmk/rosetta_inputs/8hpu_M_N_A.pdb'
pose = pyrosetta.pose_from_pdb(pdb)

print('总残基数:', pose.total_residue())
print('链数:', pose.num_chains())
print('前 30 个残基:', pose.sequence()[:30])

总残基数: 424
链数: 3
前 30 个残基: VQLVESGGGLVQPGGSLRLSCAASEITVSS


**两套编号的对照** —— PDBInfo 是连接 Rosetta 内部编号和原始 PDB 编号的桥梁。

In [5]:
info = pose.pdb_info()

for ch in range(1, pose.num_chains() + 1):
    b, e = pose.chain_begin(ch), pose.chain_end(ch)    # 该链在 Rosetta 编号里的起止
    print(f'链 {info.chain(b)}   Rosetta {b:>4} - {e:<4}   PDB {info.number(b):>4} - {info.number(e):<4}')

链 H   Rosetta    1 - 121    PDB    1 - 121 
链 L   Rosetta  122 - 230    PDB    1 - 109 
链 A   Rosetta  231 - 424    PDB    1 - 194 


可以看到每条链的 PDB 编号都从头开始，而 Rosetta 编号一路连续往下数。

下面是双向换算，**以后选残基必用**。

In [6]:
print('L 链第 30 位  ->  Rosetta 编号', info.pdb2pose('L', 30))    # PDB -> Rosetta
print('Rosetta 150   ->  PDB', info.pose2pdb(150))               # Rosetta -> PDB

i = info.pdb2pose('L', 30)
print('核对:', pose.residue(i).name3(), '在', info.pose2pdb(i))

L 链第 30 位  ->  Rosetta 编号 151
Rosetta 150   ->  PDB 29 L 
核对: SER 在 30 L 


**多链的 FoldTree** —— 链内是肽链边（`-1`），链间靠 `jump` 连接。
InterfaceAnalyzer 把两条链拉开算结合能，走的就是 jump。

In [7]:
print(pose.fold_tree())

FOLD_TREE  EDGE 1 121 -1  EDGE 1 122 1  EDGE 122 230 -1  EDGE 1 231 2  EDGE 231 424 -1 


In [8]:
chains = pose.split_by_chain()    # 按链拆成独立的 Pose

for k in range(1, len(chains) + 1):
    print(k, chains[k].total_residue(), chains[k].sequence()[:20])

1 121 VQLVESGGGLVQPGGSLRLS
2 109 DIQMTQSPSSLSASVGDRVS
3 194 NLCPFDEVFNATRFASVYAW
